# Gradient Boosting — Banknote Authentication

We use `rice_ml`'s binary gradient boosting classifier on the banknote authentication data. Each shallow regression tree is fit to the pseudo-residual $y - \sigma(F)$ of the current log-odds ensemble $F$.

**Update:** $\;F_m(x) = F_{m-1}(x) + \eta\, h_m(x)$, where $h_m$ approximates the negative gradient of the deviance.

This notebook now goes past a basic train/test score by adding exploratory checks, hyperparameter comparisons, staged loss curves, threshold tuning, permutation importance, a decision-region view, and examples of the most uncertain predictions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rice_ml.processing.datasets import find_data_file

np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.5)

In [2]:
from rice_ml.supervised_learning.decision_trees import DecisionTreeClassifier
from rice_ml.supervised_learning.ensemble_methods import (
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from rice_ml.processing.pre_processing import StandardScaler, train_test_split
from rice_ml.processing.post_processing import (
    accuracy_score,
    confusion_matrix,
    precision_recall_f1,
    roc_auc_score,
)

feature_names = ["variance", "skewness", "curtosis", "entropy"]

df = pd.read_csv(find_data_file("BankNote_Authentication.csv"))
X = df[feature_names].to_numpy(dtype=float)
y = df["class"].to_numpy(dtype=int)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0,
)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

## 1. Explore the data first

The four input columns are wavelet-transform statistics from banknote images. Before fitting the ensemble, check class balance and how the classes separate in two common feature views.

In [ ]:
print("Shape:", df.shape)
print("Class counts:")
print(df["class"].value_counts().sort_index())
print("\nFeature summary by class:")
print(df.groupby("class")[feature_names].mean().round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, x_col, y_col in [
    (axes[0], "variance", "skewness"),
    (axes[1], "curtosis", "entropy"),
]:
    for cls, label, color in [(0, "class 0", "tab:blue"), (1, "class 1", "tab:orange")]:
        part = df[df["class"] == cls]
        ax.scatter(part[x_col], part[y_col], s=18, alpha=0.65, label=label, color=color)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f"{x_col} vs. {y_col}")
axes[0].legend()
plt.tight_layout()
plt.show()

## 2. Compare against simpler baselines

A gradient boosted ensemble should beat a very shallow tree and compete with a random forest. Accuracy is useful, but ROC-AUC also checks whether the probability ranking is strong.

In [3]:
gb = GradientBoostingClassifier(
    n_estimators=120, learning_rate=0.1, max_depth=3, random_state=0,
).fit(X_tr_s, y_tr)
rf = RandomForestClassifier(
    n_estimators=100, max_depth=6, random_state=0,
).fit(X_tr_s, y_tr)
stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_tr_s, y_tr)

rows = {
    "decision stump": stump.score(X_te_s, y_te),
    "random forest": rf.score(X_te_s, y_te),
    "gradient boosting": gb.score(X_te_s, y_te),
}
print("Test accuracy:", pd.Series(rows).round(4))
print("GB ROC-AUC:", round(roc_auc_score(y_te, gb.predict_proba(X_te_s)[:, 1]), 4))

Test accuracy: decision stump       0.8480
random forest        0.9942
gradient boosting    0.9854
dtype: float64
GB ROC-AUC: 0.9991


## 3. Try several boosting settings

Gradient boosting has a learning-rate/number-of-trees tradeoff. Smaller learning rates usually need more trees, while deeper trees can capture interactions but may overfit.

In [ ]:
configs = [
    {"n_estimators": 40, "learning_rate": 0.20, "max_depth": 1},
    {"n_estimators": 80, "learning_rate": 0.10, "max_depth": 2},
    {"n_estimators": 120, "learning_rate": 0.10, "max_depth": 3},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 2},
]

results = []
for params in configs:
    model = GradientBoostingClassifier(random_state=0, **params).fit(X_tr_s, y_tr)
    proba = model.predict_proba(X_te_s)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results.append({
        **params,
        "accuracy": accuracy_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
    })

pd.DataFrame(results).sort_values(["roc_auc", "accuracy"], ascending=False).round(4)

## 4. Watch training and test loss by stage

The ensemble adds one tree at a time, so we can replay the stages and see where hold-out deviance stops improving.

In [ ]:
def staged_deviance(model, X_scaled, y_true):
    running = np.full(X_scaled.shape[0], model.init_, dtype=float)
    losses = []
    for tree in model.estimators_:
        running += model.learning_rate * tree.predict(X_scaled)
        p1 = 1.0 / (1.0 + np.exp(-running))
        p1 = np.clip(p1, 1e-15, 1 - 1e-15)
        losses.append(-np.mean(y_true * np.log(p1) + (1 - y_true) * np.log(1 - p1)))
    return np.array(losses)

train_curve = staged_deviance(gb, X_tr_s, y_tr)
test_curve = staged_deviance(gb, X_te_s, y_te)
best_stage = int(np.argmin(test_curve)) + 1

fig, ax = plt.subplots()
ax.plot(train_curve, label="train")
ax.plot(test_curve, label="test")
ax.axvline(best_stage, color="black", linestyle="--", linewidth=1, label=f"best test stage = {best_stage}")
ax.set_xlabel("# of trees")
ax.set_ylabel("Bernoulli deviance")
ax.set_title("Gradient boosting loss over stages")
ax.legend()
plt.show()

print(f"Lowest test deviance: {test_curve.min():.4f} at tree {best_stage}")

## 5. Tune the probability threshold

The default 0.50 cutoff is not always the best choice. This sweep shows how accuracy, precision, recall, and F1 move as the cutoff changes.

In [ ]:
proba_te = gb.predict_proba(X_te_s)[:, 1]
threshold_rows = []
for threshold in np.linspace(0.1, 0.9, 17):
    pred = (proba_te >= threshold).astype(int)
    prf = precision_recall_f1(y_te, pred, positive_label=1)
    threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_te, pred),
        **prf,
    })

threshold_table = pd.DataFrame(threshold_rows)
best_f1 = threshold_table.loc[threshold_table["f1"].idxmax()]
print("Best F1 row:")
print(best_f1.round(4))

fig, ax = plt.subplots()
for col in ["accuracy", "precision", "recall", "f1"]:
    ax.plot(threshold_table["threshold"], threshold_table[col], marker="o", label=col)
ax.set_xlabel("classification threshold")
ax.set_ylabel("score")
ax.set_ylim(0, 1.02)
ax.set_title("Threshold tradeoffs")
ax.legend()
plt.show()

pred_05 = (proba_te >= 0.5).astype(int)
print("Confusion matrix at threshold 0.50")
print(pd.DataFrame(
    confusion_matrix(y_te, pred_05, labels=[0, 1]),
    index=["actual 0", "actual 1"],
    columns=["pred 0", "pred 1"],
))

## 6. Estimate feature importance by permutation

Because the model is an ensemble, a simple model-agnostic test is to shuffle one feature at a time and measure how much test accuracy drops.

In [ ]:
def permutation_importance(model, X_scaled, y_true, names, n_repeats=30, random_state=0):
    rng = np.random.default_rng(random_state)
    baseline = model.score(X_scaled, y_true)
    rows = []
    for j, name in enumerate(names):
        drops = []
        for _ in range(n_repeats):
            X_perm = X_scaled.copy()
            X_perm[:, j] = rng.permutation(X_perm[:, j])
            drops.append(baseline - model.score(X_perm, y_true))
        rows.append({
            "feature": name,
            "mean_accuracy_drop": np.mean(drops),
            "std": np.std(drops),
        })
    return pd.DataFrame(rows).sort_values("mean_accuracy_drop", ascending=False)

importance = permutation_importance(gb, X_te_s, y_te, feature_names)
print(importance.round(4))

fig, ax = plt.subplots()
ax.barh(importance["feature"], importance["mean_accuracy_drop"], xerr=importance["std"])
ax.invert_yaxis()
ax.set_xlabel("accuracy drop after shuffling")
ax.set_title("Permutation importance")
plt.show()

## 7. Plot a two-feature decision region

The full model uses all four features. For visualization, fit a separate model on the two most important features and draw the learned probability surface.

In [ ]:
top_two = importance["feature"].head(2).tolist()
top_idx = [feature_names.index(name) for name in top_two]

viz_model = GradientBoostingClassifier(
    n_estimators=120, learning_rate=0.1, max_depth=2, random_state=0,
).fit(X_tr_s[:, top_idx], y_tr)

x_min, x_max = X_tr_s[:, top_idx[0]].min() - 0.5, X_tr_s[:, top_idx[0]].max() + 0.5
y_min, y_max = X_tr_s[:, top_idx[1]].min() - 0.5, X_tr_s[:, top_idx[1]].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = viz_model.predict_proba(grid)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5.5))
contour = ax.contourf(xx, yy, zz, levels=np.linspace(0, 1, 21), cmap="RdBu_r", alpha=0.75)
ax.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=1.2)
for cls, label, color in [(0, "class 0", "tab:blue"), (1, "class 1", "tab:orange")]:
    mask = y_te == cls
    ax.scatter(X_te_s[mask, top_idx[0]], X_te_s[mask, top_idx[1]], s=24, edgecolor="white", linewidth=0.4, label=label, color=color)
ax.set_xlabel(f"{top_two[0]} (scaled)")
ax.set_ylabel(f"{top_two[1]} (scaled)")
ax.set_title("Two-feature boosted decision region")
fig.colorbar(contour, ax=ax, label="P(class 1)")
ax.legend()
plt.show()

## 8. Inspect the hardest cases

The predictions nearest 0.50 are the least confident. Looking at them is a good way to connect the model's probabilities back to the raw feature values.

In [ ]:
uncertain_idx = np.argsort(np.abs(proba_te - 0.5))[:10]
uncertain = pd.DataFrame(X_te[uncertain_idx], columns=feature_names)
uncertain["actual"] = y_te[uncertain_idx]
uncertain["P(class 1)"] = proba_te[uncertain_idx]
uncertain["pred"] = (proba_te[uncertain_idx] >= 0.5).astype(int)
uncertain["correct"] = uncertain["actual"] == uncertain["pred"]
uncertain.round(4)

## Takeaways

- Gradient boosting often improves on a single tree and can match or beat random forests on structured classification.
- `learning_rate` × `n_estimators` trades bias against compute; shallow trees (`max_depth` 2-4) are typical.
- Monitoring both training and out-of-sample deviance shows whether extra stages are still helping.
- Probability thresholds matter when precision and recall have different costs.
- Permutation importance and uncertain-case inspection make the model easier to explain than a single final accuracy score.